In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import numpy as np
import os
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from tqdm import tqdm

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from tqdm import tqdm

def analyze_token_lengths(parquet_paths, dataset_name):
    """
    Analyze token lengths from parquet files containing tokenized text.
    
    Args:
        parquet_paths (list): List of paths to parquet files
        dataset_name (str): Name of the dataset (for display purposes)
        
    Returns:
        dict: Dictionary containing statistics about token lengths
    """
    print("Loading GPT-2 tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    all_lengths = []
    
    for path in parquet_paths:
        try:
            print(f"\nProcessing: {path}")
            df = pd.read_parquet(path)
            print(f"Columns in file: {df.columns.tolist()}")
            
            if 'input_ids' not in df.columns:
                print("Error: 'input_ids' column not found in the parquet file")
                continue
            
            # Process in batches to handle large files
            batch_size = 1000
            total_batches = (len(df) + batch_size - 1) // batch_size
            
            for i in tqdm(range(0, len(df), batch_size), desc=f"Processing {os.path.basename(path)}"):
                batch = df.iloc[i:i+batch_size]
                # Get token lengths from input_ids
                lengths = batch['input_ids'].apply(lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0)
                all_lengths.extend(lengths.tolist())
            
            print(f"Processed {len(df)} examples")
            
        except Exception as e:
            print(f"Error processing {path}: {e}")
            import traceback
            traceback.print_exc()
    
    if not all_lengths:
        print("No valid token lengths found in any files")
        return None
        
    # Calculate statistics
    all_lengths = np.array(all_lengths)
    return {
        'dataset': dataset_name,
        'mean': float(np.mean(all_lengths)),
        'median': float(np.median(all_lengths)),
        'std': float(np.std(all_lengths)),
        'min': int(np.min(all_lengths)),
        'max': int(np.max(all_lengths)),
        'percentiles': {
            '25': float(np.percentile(all_lengths, 25)),
            '50': float(np.percentile(all_lengths, 50)),
            '75': float(np.percentile(all_lengths, 75)),
            '90': float(np.percentile(all_lengths, 90)),
            '95': float(np.percentile(all_lengths, 95)),
            '99': float(np.percentile(all_lengths, 99))
        },
        'total_sequences': len(all_lengths),
        'total_tokens': int(np.sum(all_lengths))
    }

def plot_length_distribution(stats_list, output_dir='output'):
    """Plot the distribution of sequence lengths."""
    os.makedirs(output_dir, exist_ok=True)
    
    for stats in stats_list:
        if stats is None:
            continue
            
        plt.figure(figsize=(12, 6))
        plt.hist(stats['lengths'], bins=50, alpha=0.7, label=stats['dataset'])
        plt.title(f"Sequence Length Distribution - {stats['dataset']}")
        plt.xlabel('Sequence Length (tokens)')
        plt.ylabel('Frequency')
        plt.axvline(stats['mean'], color='r', linestyle='dashed', linewidth=1, 
                   label=f'Mean: {stats["mean"]:.1f}')
        plt.axvline(stats['median'], color='g', linestyle='dashed', linewidth=1,
                   label=f'Median: {stats["median"]:.1f}')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Save the plot
        safe_name = "".join(c if c.isalnum() else "_" for c in stats['dataset'])
        plt.savefig(f"{output_dir}/{safe_name}_length_distribution.png")
        plt.close()

def main():
    # Base directory containing the English data
    base_dir_en = "/home/s2587034/pretrained_checkpoints/babylm_models_en/babylm_shuffle_control_100M_randinit/babylm_shuffle_control_100M_randinit_seed0/artifacts/gpt2-processed/babylm_shuffle_control_100M_seed0-/home/s2587034/dissertation/code/training/babylm_dataset.py/preprocessing/tokenization"
    
    # Paths to English data
    en_train_paths = [
        f"{base_dir_en}/train-tokenized/docs-0.parquet",
        f"{base_dir_en}/train-tokenized/docs-1.parquet"
    ]
    en_val_path = f"{base_dir_en}/validation-tokenized/docs-0.parquet"
    
    # Analyze English data
    print("Analyzing English training data...")
    en_train_stats = analyze_token_lengths(en_train_paths, "English Train")
    print("\nAnalyzing English validation data...")
    en_val_stats = analyze_token_lengths([en_val_path], "English Validation")
    
    # Print statistics
    print("\n=== Statistics ===")
    for stats in [en_train_stats, en_val_stats]:
        if stats:
            print(f"\n{stats['dataset']}:")
            print(f"  Total sequences: {stats['total_sequences']:,}")
            print(f"  Total tokens: {stats['total_tokens']:,}")
            print(f"  Mean length: {stats['mean']:.2f} tokens")
            print(f"  Median length: {stats['median']:.2f} tokens")
            print(f"  Min/Max: {stats['min']} / {stats['max']} tokens")
            print(f"  Standard deviation: {stats['std']:.2f}")
            print("  Percentiles (25/50/75/90/95/99):", 
                  "/".join(f"{int(v)}" for k, v in stats['percentiles'].items()))
    
    # Prepare data for plotting
    stats_list = []
    if en_train_stats:
        stats_list.append({
            'dataset': en_train_stats['dataset'],
            'lengths': np.random.choice(
                np.arange(len(en_train_stats['lengths'])), 
                size=min(10000, len(en_train_stats['lengths'])),  # Sample for plotting
                replace=False
            ).tolist()
        })
    
    if en_val_stats:
        stats_list.append({
            'dataset': en_val_stats['dataset'],
            'lengths': en_val_stats['lengths']
        })
    
    # Plot distributions
    if stats_list:
        plot_length_distribution(stats_list)
        print("\nPlots saved to the 'output' directory")

def analyze_arrow_file(file_path):
    """
    Analyze an Arrow file and print its structure and content.
    
    Args:
        file_path (str): Path to the Arrow file
    """
    print(f"Analyzing file: {file_path}")
    print("-" * 80)
    
    # Check file size
    file_size = os.path.getsize(file_path) / (1024 * 1024)  # in MB
    print(f"File size: {file_size:.2f} MB")
    
    try:
        # Open the Arrow file
        table = pa.ipc.RecordBatchFileReader(pa.memory_map(file_path, 'r')).read_all()
        
        # Print schema information
        print("\n=== Schema ===")
        print(table.schema)
        
        # Get column names and types
        print("\n=== Columns ===")
        for field in table.schema:
            print(f"- {field.name}: {field.type}")
        
        # Show first few rows
        print("\n=== First 5 rows ===")
        df = table.to_pandas()
        print(df.head())
        
        # Basic statistics for each column
        print("\n=== Basic Statistics ===")
        print(df.describe(include='all'))
        
        # If there's a text column, analyze text lengths
        text_columns = [col for col in df.columns if df[col].dtype == 'object']
        for col in text_columns:
            print(f"\n=== Text Length Analysis for '{col}' ===")
            df[f'{col}_length'] = df[col].str.len()
            print(df[f'{col}_length'].describe().to_string())
            
            # Sample some text
            print("\nSample text:")
            for i, text in enumerate(df[col].head(3).tolist(), 1):
                print(f"{i}. {text[:200]}..." if len(str(text)) > 200 else f"{i}. {text}")
        
        # Memory usage
        print("\n=== Memory Usage ===")
        print(df.memory_usage(deep=True).to_string())
        
    except Exception as e:
        print(f"Error analyzing file: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    # Path to your Arrow file
    arrow_file = "/home/s2587034/pretrained_checkpoints/babylm_models_en/babylm_shuffle_control_100M_randinit/babylm_shuffle_control_100M_randinit_seed0/artifacts/datasets/baby_lm_corpus/babylm_shuffle_control_100M_seed0/0.0.0/5271a8e652c7ffb6ee774e1c7851e4ee84c58bc953ab0c7b66d48f64deb72b59/baby_lm_corpus-train.arrow"
    
    analyze_arrow_file(arrow_file)


Analyzing file: /home/s2587034/pretrained_checkpoints/babylm_models_en/babylm_shuffle_control_100M_randinit/babylm_shuffle_control_100M_randinit_seed0/artifacts/datasets/baby_lm_corpus/babylm_shuffle_control_100M_seed0/0.0.0/5271a8e652c7ffb6ee774e1c7851e4ee84c58bc953ab0c7b66d48f64deb72b59/baby_lm_corpus-train.arrow
--------------------------------------------------------------------------------
File size: 495.68 MB
Error analyzing file: Not an Arrow file


Traceback (most recent call last):
  File "/tmp/ipykernel_600454/3723047665.py", line 180, in analyze_arrow_file
    table = pa.ipc.RecordBatchFileReader(pa.memory_map(file_path, 'r')).read_all()
            ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/s2587034/.conda/envs/dissertation/lib/python3.13/site-packages/pyarrow/ipc.py", line 110, in __init__
    self._open(source, footer_offset=footer_offset,
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
               options=options, memory_pool=memory_pool)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "pyarrow/ipc.pxi", line 1090, in pyarrow.lib._RecordBatchFileReader._open
  File "pyarrow/error.pxi", line 155, in pyarrow.lib.pyarrow_internal_check_status
  File "pyarrow/error.pxi", line 92, in pyarrow.lib.check_status
pyarrow.lib.ArrowInvalid: Not an Arrow file


In [7]:
from datasets import load_from_disk, Dataset, DatasetDict
import os
import pandas as pd
from tqdm import tqdm

def analyze_hf_dataset(dataset_path):
    """
    Analyze a Hugging Face dataset saved on disk.
    
    Args:
        dataset_path (str): Path to the dataset directory
    """
    print(f"Analyzing dataset at: {dataset_path}")
    print("-" * 80)
    
    try:
        # Try to load the dataset
        print("Loading dataset...")
        dataset = load_from_disk(dataset_path)
        
        # Check if it's a DatasetDict or Dataset
        if isinstance(dataset, DatasetDict):
            print("\n=== Dataset Split Information ===")
            for split_name, split_data in dataset.items():
                print(f"\nSplit: {split_name}")
                print(f"Number of examples: {len(split_data):,}")
                print("Features:", split_data.features)
                
                # Show first few examples
                print("\nFirst few examples:")
                for i, example in enumerate(split_data.select(range(min(3, len(split_data))))):
                    print(f"\nExample {i+1}:")
                    for key, value in example.items():
                        if isinstance(value, str):
                            print(f"  {key}: {value[:200]}..." if len(value) > 200 else f"  {key}: {value}")
                        else:
                            print(f"  {key}: {type(value)} (length: {len(value) if hasattr(value, '__len__') else 'N/A'})")
                
        elif isinstance(dataset, Dataset):
            print("\n=== Dataset Information ===")
            print(f"Number of examples: {len(dataset):,}")
            print("Features:", dataset.features)
            
            # Show first few examples
            print("\nFirst few examples:")
            for i, example in enumerate(dataset.select(range(min(3, len(dataset))))):
                print(f"\nExample {i+1}:")
                for key, value in example.items():
                    if isinstance(value, str):
                        print(f"  {key}: {value[:200]}..." if len(value) > 200 else f"  {key}: {value}")
                    else:
                        print(f"  {key}: {type(value)} (length: {len(value) if hasattr(value, '__len__') else 'N/A'})")
            
            # Basic statistics for text columns
            text_columns = [name for name, feature in dataset.features.items() 
                          if feature.dtype == 'string' or (hasattr(feature, 'dtype') and 'str' in str(feature.dtype))]
            
            if text_columns:
                print("\n=== Text Statistics ===")
                for col in text_columns:
                    print(f"\nColumn: {col}")
                    # Get lengths
                    lengths = [len(str(x).split()) if x is not None else 0 for x in dataset[col]]
                    lengths_series = pd.Series(lengths)
                    print("Word count statistics:")
                    print(lengths_series.describe().to_string())
                    
                    # Show some examples
                    print("\nSample text:")
                    for i, text in enumerate(dataset.select_columns([col])[:3][col], 1):
                        text = str(text)
                        print(f"{i}. {text[:200]}..." if len(text) > 200 else f"{i}. {text}")
        
        else:
            print("Unknown dataset format:", type(dataset))
            
    except Exception as e:
        print(f"Error analyzing dataset: {e}")
        import traceback
        traceback.print_exc()
        
        # Try to read as parquet
        print("\nTrying to read as parquet...")
        try:
            import pyarrow.parquet as pq
            table = pq.read_table(dataset_path)
            print("\nSuccessfully read as parquet file!")
            print("Schema:", table.schema)
            print("\nFirst few rows:")
            print(table.to_pandas().head())
        except Exception as e2:
            print("Failed to read as parquet:", e2)

if __name__ == "__main__":
    # Path to your dataset
    dataset_path = "/home/s2587034/pretrained_checkpoints/babylm_models_en/babylm_shuffle_control_100M_randinit/babylm_shuffle_control_100M_randinit_seed0/artifacts/datasets/baby_lm_corpus/babylm_shuffle_control_100M_seed0/0.0.0/5271a8e652c7ffb6ee774e1c7851e4ee84c58bc953ab0c7b66d48f64deb72b59"
    
    analyze_hf_dataset(dataset_path)

Analyzing dataset at: /home/s2587034/pretrained_checkpoints/babylm_models_en/babylm_shuffle_control_100M_randinit/babylm_shuffle_control_100M_randinit_seed0/artifacts/datasets/baby_lm_corpus/babylm_shuffle_control_100M_seed0/0.0.0/5271a8e652c7ffb6ee774e1c7851e4ee84c58bc953ab0c7b66d48f64deb72b59
--------------------------------------------------------------------------------
Loading dataset...
Error analyzing dataset: Directory /home/s2587034/pretrained_checkpoints/babylm_models_en/babylm_shuffle_control_100M_randinit/babylm_shuffle_control_100M_randinit_seed0/artifacts/datasets/baby_lm_corpus/babylm_shuffle_control_100M_seed0/0.0.0/5271a8e652c7ffb6ee774e1c7851e4ee84c58bc953ab0c7b66d48f64deb72b59 is neither a `Dataset` directory nor a `DatasetDict` directory.

Trying to read as parquet...
Failed to read as parquet: Error creating dataset. Could not read schema from '/home/s2587034/pretrained_checkpoints/babylm_models_en/babylm_shuffle_control_100M_randinit/babylm_shuffle_control_100M_ra

Traceback (most recent call last):
  File "/tmp/ipykernel_600454/3343008940.py", line 19, in analyze_hf_dataset
    dataset = load_from_disk(dataset_path)
  File "/home/s2587034/.conda/envs/dissertation/lib/python3.13/site-packages/datasets/load.py", line 2148, in load_from_disk
    raise FileNotFoundError(
        f"Directory {dataset_path} is neither a `Dataset` directory nor a `DatasetDict` directory."
    )
FileNotFoundError: Directory /home/s2587034/pretrained_checkpoints/babylm_models_en/babylm_shuffle_control_100M_randinit/babylm_shuffle_control_100M_randinit_seed0/artifacts/datasets/baby_lm_corpus/babylm_shuffle_control_100M_seed0/0.0.0/5271a8e652c7ffb6ee774e1c7851e4ee84c58bc953ab0c7b66d48f64deb72b59 is neither a `Dataset` directory nor a `DatasetDict` directory.
